#### 6 - Evaluate NLP Classifier

##### Purpose

This notebook performs the final evaluation of the TF-IDF + Logistic Regression baseline.

It:

- reads the persisted modeling dataset
- reuses the existing train/validation/test assignments
- fits TF-IDF on training text only
- trains Logistic Regression on training data only
- evaluates the trained model on the untouched test set
- calculates multiclass metrics
- builds a confusion matrix
- produces a classification report
- inspects class probabilities and misclassified tickets
- compares training, validation, and test performance
- documents the limitations of the current small dataset

No model registration or deployment is performed here.


##### 1. Architecture

``` text

nlp_modeling_dataset
        ↓
dataset_split
        ↓
 ┌──────────────┬────────────────┬──────────────┐
 ↓              ↓                ↓
Train        Validation          Test
 ↓              ↓                ↓
TF-IDF fit      transform        transform
+ transform
 ↓
Logistic Regression
 ↓
model.fit()
 ↓
trained classifier
        ↓
        ├── Training evaluation
        ├── Validation evaluation
        └── Final Test evaluation

```


##### 2. Technologies

- Python
- PySpark
- pandas
- NumPy
- scikit-learn
- TF-IDF
- Logistic Regression
- Classification Metrics
- Confusion Matrix
- Unity Catalog
- Delta Lake


##### 3. Imports

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
)

from sklearn.linear_model import (
    LogisticRegression,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from src.project_config import (
    MODELING_TABLE,
    TICKET_ID_COL,
    CLEAN_TEXT_COL,
    TARGET_COL,
    SPLIT_COL,
    EXPECTED_CATEGORIES,
    RANDOM_SEED,
    TFIDF_MAX_FEATURES,
    TFIDF_NGRAM_RANGE,
    TFIDF_MIN_DF,
    TFIDF_MAX_DF,
)

##### 4. Load Modeling Dataset

In [0]:
modeling_df = spark.table(
    MODELING_TABLE
)

In [0]:
display(
    modeling_df.limit(5)
)

In [0]:
modeling_row_count = (
    modeling_df.count()
)

print(
    f"Modeling rows: "
    f"{modeling_row_count:,}"
)

In [0]:
if modeling_row_count == 0:
    raise ValueError(
        f"Modeling table contains no records: "
        f"{MODELING_TABLE}"
    )

##### 5. Validate Input Contract

In [0]:
required_columns = {
    TICKET_ID_COL,
    CLEAN_TEXT_COL,
    TARGET_COL,
    SPLIT_COL,
}

missing_columns = (
    required_columns
    - set(modeling_df.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        f"{sorted(missing_columns)}"
    )

print(
    "Input contract validation passed."
)

##### 6. Create Train / Validation / Test DataFrames

In [0]:
train_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "train"
    )
)

validation_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "validation"
    )
)

test_df = (
    modeling_df
    .filter(
        F.col(SPLIT_COL) == "test"
    )
)

In [0]:
train_count = train_df.count()
validation_count = validation_df.count()
test_count = test_df.count()

print(f"Train rows      : {train_count:,}")
print(f"Validation rows : {validation_count:,}")
print(f"Test rows       : {test_count:,}")

##### 7. Validate Category Coverage

In [0]:
split_category_distribution_df = (
    modeling_df
    .groupBy(
        SPLIT_COL,
        TARGET_COL,
    )
    .count()
    .orderBy(
        SPLIT_COL,
        TARGET_COL,
    )
)

display(
    split_category_distribution_df
)

##### 8. Convert Required Columns to pandas

In [0]:
train_pdf = (
    train_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

validation_pdf = (
    validation_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

test_pdf = (
    test_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

##### 9. Separate Features and Targets

In [0]:
X_train_text = (
    train_pdf[
        CLEAN_TEXT_COL
    ]
)

y_train = (
    train_pdf[
        TARGET_COL
    ]
)

X_validation_text = (
    validation_pdf[
        CLEAN_TEXT_COL
    ]
)

y_validation = (
    validation_pdf[
        TARGET_COL
    ]
)

X_test_text = (
    test_pdf[
        CLEAN_TEXT_COL
    ]
)

y_test = (
    test_pdf[
        TARGET_COL
    ]
)

##### 10. Create TF-IDF Vectorizer

In [0]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
)

##### 11. Fit TF-IDF on Training Data Only

In [0]:
X_train_tfidf = (
    tfidf_vectorizer
    .fit_transform(
        X_train_text
    )
)

In [0]:
X_validation_tfidf = (
    tfidf_vectorizer
    .transform(
        X_validation_text
    )
)

In [0]:
X_test_tfidf = (
    tfidf_vectorizer
    .transform(
        X_test_text
    )
)

In [0]:
print(
    "Train:",
    X_train_tfidf.shape,
)

print(
    "Validation:",
    X_validation_tfidf.shape,
)

print(
    "Test:",
    X_test_tfidf.shape,
)

##### 12. Train Logistic Regression

In [0]:
classifier = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_SEED,
)

In [0]:
classifier.fit(
    X_train_tfidf,
    y_train,
)

In [0]:
print(
    classifier.classes_
)

##### 13. Generate Predictions

In [0]:
train_predictions = (
    classifier.predict(
        X_train_tfidf
    )
)

In [0]:
validation_predictions = (
    classifier.predict(
        X_validation_tfidf
    )
)

In [0]:
test_predictions = (
    classifier.predict(
        X_test_tfidf
    )
)

##### 14. Accuracy

In [0]:
test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

print(
    f"Test accuracy: "
    f"{test_accuracy:.4f}"
)

##### 15. Precision

In [0]:
#Of everything predicted as this class, how many predictions were actually correct?

In [0]:
test_precision_macro = (
    precision_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    )
)

print(
    f"Test macro precision: "
    f"{test_precision_macro:.4f}"
)

Predicted Billing = 5 tickets

Actually Billing = 4

Precision = 4 / 5

##### 16. Recall

- Of all tickets that really belong to this class, how many did the model find?
- Actually Billing = 5
- Correctly predicted Billing = 4
- Recall = 4 / 5

In [0]:
test_recall_macro = (
    recall_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    )
)

print(
    f"Test macro recall: "
    f"{test_recall_macro:.4f}"
)

##### 17. F1 Score

Precision and recall can move in different directions.

F1 combines them:

high precision
+
high recall
        ↓
high F1

In [0]:
test_f1_macro = (
    f1_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    )
)

print(
    f"Test macro F1: "
    f"{test_f1_macro:.4f}"
)

##### 18. Macro vs Weighted Metrics

In [0]:
test_f1_weighted = (
    f1_score(
        y_test,
        test_predictions,
        average="weighted",
        zero_division=0,
    )
)

print(
    f"Test weighted F1: "
    f"{test_f1_weighted:.4f}"
)

This is particularly important for multiclass problems.

###### Macro

Macro treats every category equally.

Conceptually:

``` text

F1 Billing
+
F1 Cancellation
+
F1 Login
+
F1 Technical
----------------
        4
```
Even if one category has fewer examples, it gets the same importance.

###### Weighted

Weighted metrics account for how many examples each class has.

Conceptually:

``` text

Class metric
    ×
number of examples in that class

```
So:

``` text

macro
    ↓
every class gets equal importance

weighted
    ↓
larger classes contribute more

```

##### 19. Calculate Complete Metric Summary

In [0]:
test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),

    "precision_macro": precision_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    ),

    "recall_macro": recall_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    ),

    "f1_macro": f1_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    ),

    "precision_weighted": precision_score(
        y_test,
        test_predictions,
        average="weighted",
        zero_division=0,
    ),

    "recall_weighted": recall_score(
        y_test,
        test_predictions,
        average="weighted",
        zero_division=0,
    ),

    "f1_weighted": f1_score(
        y_test,
        test_predictions,
        average="weighted",
        zero_division=0,
    ),
}

In [0]:
metrics_df = pd.DataFrame(
    [
        {
            "metric": metric,
            "value": value,
        }
        for metric, value
        in test_metrics.items()
    ]
)

display(
    metrics_df
)

##### 20. Classification Report

In [0]:
#classification_report() gives per-class metrics.
report_dict = (
    classification_report(
        y_test,
        test_predictions,
        labels=list(
            classifier.classes_
        ),
        output_dict=True,
        zero_division=0,
    )
)

In [0]:
classification_report_df = (
    pd.DataFrame(
        report_dict
    )
    .transpose()
    .rename_axis(
        "class"
    )
    .reset_index()
)

display(
    classification_report_df
)

##### 21. What Does support Mean?

In a classification report: support, means: How many true examples of this class exist in the evaluated dataset?

For example:

Billing
support = 4

means there were four actual Billing tickets in the test set.

It is not a model score.

##### 22. Build the Confusion Matrix

In [0]:
class_labels = list(
    classifier.classes_
)

confusion = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_labels,
)

In [0]:
confusion_df = (
    pd.DataFrame(
        confusion,
        index=class_labels,
        columns=[
            f"Predicted_{label}"
            for label in class_labels
        ],
    )
    .rename_axis(
        "Actual_Category"
    )
    .reset_index()
)

In [0]:
display(
    confusion_df
)

##### 23. How to Read the Confusion Matrix

Conceptually:

``` text


                 PREDICTED

              Bill Cancel Login Tech

ACTUAL Bill      3     0     0    0

       Cancel    0     3     0    0

       Login     0     0     3    0

       Tech      0     0     0    4
```

Values on the diagonal are correct predictions.

``` text

Actual Billing
Predicted Billing
        ↓
correct

```
Values outside the diagonal are mistakes.


For example:
``` text

Actual Login
Predicted Technical
        ↓
misclassification

```

The confusion matrix tells us which categories are being confused with which other categories.

##### 24. Build Detailed Test Results

In [0]:
test_results_df = (
    test_pdf[
        [
            TICKET_ID_COL,
            CLEAN_TEXT_COL,
            TARGET_COL,
        ]
    ]
    .copy()
)

test_results_df[
    "predicted_category"
] = (
    test_predictions
)

test_results_df[
    "is_correct"
] = (
    test_results_df[
        TARGET_COL
    ]
    ==
    test_results_df[
        "predicted_category"
    ]
)

In [0]:
display(
    test_results_df
)

##### 25. Inspect Test Errors

In [0]:
test_errors_df = (
    test_results_df[
        ~test_results_df[
            "is_correct"
        ]
    ]
    .copy()
)

In [0]:
test_error_count = len(
    test_errors_df
)

print(
    f"Test misclassifications: "
    f"{test_error_count}"
)

In [0]:
display(
    test_errors_df
)

##### 26. Generate Test Probabilities

In [0]:
test_probabilities = (
    classifier.predict_proba(
        X_test_tfidf
    )
)

In [0]:
print(
    test_probabilities.shape
)

##### 27. Add Prediction Confidence

probability
↓
one value PER CLASS


confidence
↓
highest of those probabilities

In [0]:
test_results_df[
    "prediction_confidence"
] = (
    test_probabilities.max(
        axis=1
    )
)

In [0]:
display(
    test_results_df
    .sort_values(
        "prediction_confidence",
        ascending=True,
    )
)

##### 28. Inspect Full Probability Distribution

In [0]:
for class_index, class_name in enumerate(
    classifier.classes_
):

    probability_column = (
        f"prob_{class_name.lower()}"
    )

    test_results_df[
        probability_column
    ] = (
        test_probabilities[
            :,
            class_index,
        ]
    )

In [0]:
display(
    test_results_df
)

##### 29. Compare Training, Validation, and Test Metrics

In [0]:
train_accuracy = accuracy_score(
    y_train,
    train_predictions,
)

validation_accuracy = accuracy_score(
    y_validation,
    validation_predictions,
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

In [0]:
train_f1_macro = f1_score(
    y_train,
    train_predictions,
    average="macro",
    zero_division=0,
)

validation_f1_macro = f1_score(
    y_validation,
    validation_predictions,
    average="macro",
    zero_division=0,
)

test_f1_macro = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0,
)

In [0]:
performance_comparison_df = (
    pd.DataFrame(
        {
            "split": [
                "train",
                "validation",
                "test",
            ],
            "accuracy": [
                train_accuracy,
                validation_accuracy,
                test_accuracy,
            ],
            "macro_f1": [
                train_f1_macro,
                validation_f1_macro,
                test_f1_macro,
            ],
        }
    )
)

In [0]:
display(
    performance_comparison_df
)

##### 30. Interpret Performance Gaps

In [0]:
train_test_accuracy_gap = (
    train_accuracy
    - test_accuracy
)

train_test_f1_gap = (
    train_f1_macro
    - test_f1_macro
)

In [0]:
print(
    f"Train-test accuracy gap: "
    f"{train_test_accuracy_gap:.4f}"
)

print(
    f"Train-test macro F1 gap: "
    f"{train_test_f1_gap:.4f}"
)

Training much higher than test
        ↓
possible overfitting

Training and test similar
        ↓
better generalization signal

##### 31. Why Accuracy Alone Is Not Enough

Suppose the test set had:

- Billing        8
- Technical      2
- Login          2
- Cancellation   1

A model might perform well on Billing and poorly on everything else while still obtaining respectable accuracy.

That is why we inspect:

- accuracy
- precision
- recall
- F1
- macro metrics
- weighted metrics
- confusion matrix
- per-class report
- individual errors

Evaluation is not one number.

##### 32. Error Analysis

When the model is wrong, ask why.

Typical possibilities include:

- unknown vocabulary
- ambiguous ticket wording
- multiple intents in one ticket
- insufficient training examples
- overlapping category language
- generic wording
- incorrect label

For example:

"I can't access my account and want to cancel it."

could reasonably contain signals for both:

Login
Cancellation

Real NLP datasets contain many such cases.

##### 33. Check Unknown Test Words

In [0]:
#The vectorizer cannot represent words unseen during training.
training_vocabulary = set(
    tfidf_vectorizer
    .get_feature_names_out()
)

In [0]:
test_words = set(
    " ".join(
        test_pdf[
            CLEAN_TEXT_COL
        ]
    ).split()
)

In [0]:
unknown_test_words = (
    test_words
    - training_vocabulary
)

In [0]:
print(
    "Test words not represented "
    "in training vocabulary:"
)

print(
    sorted(
        unknown_test_words
    )
)

These are ignored by the TF-IDF representation.

They can help explain some model mistakes.

##### 34. Final Evaluation Summary

In [0]:
print(
    f"""
Final TF-IDF + Logistic Regression Evaluation
-----------------------------------------------

Training rows   : {train_count}
Validation rows : {validation_count}
Test rows       : {test_count}

Vocabulary size : {
    X_train_tfidf.shape[1]
}

Test Accuracy   : {test_accuracy:.4f}
Test Macro F1   : {test_f1_macro:.4f}
Test Weighted F1: {test_f1_weighted:.4f}
Test Errors     : {test_error_count}
"""
)

###### 35. Final Validation

In [0]:
assert len(
    test_predictions
) == test_count

assert (
    test_probabilities.shape
    ==
    (
        test_count,
        len(
            EXPECTED_CATEGORIES
        ),
    )
)

assert (
    confusion.shape
    ==
    (
        len(
            EXPECTED_CATEGORIES
        ),
        len(
            EXPECTED_CATEGORIES
        ),
    )
)

assert 0.0 <= test_accuracy <= 1.0
assert 0.0 <= test_f1_macro <= 1.0

print(
    "Final evaluation validation passed."
)

##### Production Design Decisions

This notebook follows these principles:

- It is independently runnable.
- It reads the persisted modeling dataset.
- Existing split assignments are reused.
- TF-IDF is fitted only on training data.
- Logistic Regression is fitted only on training data.
- Validation remains separate from final testing.
- The test set is used for final evaluation rather than model fitting.
- Evaluation includes class-level metrics rather than accuracy alone.
- A confusion matrix is used to inspect class confusion.
- Prediction probabilities are inspected for uncertainty.
- Error analysis uses the original cleaned text.
- Configuration comes from src/project_config.py.

##### Key Learnings

The complete traditional NLP baseline is now:

``` text

Raw Text
   ↓
Cleaning
   ↓
Tokenization
   ↓
Vocabulary
   ↓
TF-IDF
   ↓
Logistic Regression
   ↓
Predicted Category
   ↓
Evaluation

```

Accuracy answers: How many predictions were correct overall?

Precision asks: When the model predicts a class, how often is it correct?

Recall asks: Of all actual examples of a class,how many did the model find?

F1 balances: precision + recall

Macro metrics treat every category equally.

Weighted metrics give larger classes more influence.

The confusion matrix tells us which classes are being confused, while error analysis explains the individual tickets behind those numbers.

##### Conclusion

06_evaluate_nlp_classifier completes the traditional machine-learning portion of the project.

We now have a fully evaluated baseline:

TF-IDF
   +
Logistic Regression

This baseline becomes extremely valuable later because embeddings, neural networks, and transformers should not simply be assumed to be better.

We should ask:

Do the more advanced NLP approaches actually improve performance or capability relative to this simple baseline?

##### Next Notebook

##### 07_embeddings_foundations

The progression becomes:

- Bag of Words --> word occurrence
- TF-IDF --> word importance
- Embeddings  --> word / text meaning